In [2]:
import os
from dotenv import load_dotenv
import requests
import json 
import pandas as pd 


In [ ]:
load_dotenv()
API_KEY = os.getenv("NPS_API_KEY") # Make sure to set your NPS_API_KEY in the .env file
url = "https://developer.nps.gov/api/v1/parks" 

params = {
    "api_key": API_KEY,
    "limit": 500 # Adjust the limit as needed (max 500 per request)
}

response = requests.get(url, params=params)
data = response.json()

parks = data["data"]

rows = []

for park in parks:
    activities = [a["name"] for a in park["activities"]] # Extract activity names

    topics = [t["name"] for t in park["topics"]] # Extract topic names

    row = {
        "park_name": park["fullName"],                  # Use fullName for better readability
        "park_code": park["parkCode"],                  # Unique identifier for the park
        "states": park["states"],                       # Some parks span multiple states, so we keep this as a string
        "designation": park["designation"],             # Type of park (e.g., National Park, National Monument)
        "latitude": park["latitude"], 
        "longitude": park["longitude"],
        "num_activities": len(activities),              # Count of activities offered by the park
        "num_topics": len(topics),                      # Count of topics associated with the park
        "num_entrance_fees": len(park["entranceFees"]), # Count of entrance fee options available
        "description_length": len(park["description"]), # Length of the park description (as a proxy for how much information is available)

        "has_camping": "Camping" in activities,                                         # Check if camping is offered
        "has_fishing": "Fishing" in activities,                                         # Check if fishing is offered
        "has_wildlife": "Wildlife Watching" in activities or "Wildlife" in topics,      # Check if wildlife watching is offered (either as an activity or a topic)
        "has_biking": "Biking" in activities,                                           # Check if biking is offered    
        "has_boating": "Boating" in activities                                          # Check if boating is offered   
    }
    rows.append(row)

df = pd.DataFrame(rows) # Create a DataFrame from the list of dictionaries

# Convert numeric columns
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

# Save dataset
df.to_csv("national_parks_dataset.csv", index=False)

print("Number of parks:", len(df))
print(df.head())

Dataset created!
Number of parks: 474
                                           park_name park_code states  \
0  Abraham Lincoln Birthplace National Historical...      abli     KY   
1                               Acadia National Park      acad     ME   
2                     Adams National Historical Park      adam     MA   
3                African American Civil War Memorial      afam     DC   
4            African Burial Ground National Monument      afbg     NY   

                designation   latitude  longitude  num_activities  num_topics  \
0  National Historical Park  37.585866 -85.673305              15           9   
1             National Park  44.409286 -68.247501              46          31   
2  National Historical Park  42.255396 -71.011604               8           8   
3                            38.916600 -77.026000               2           4   
4         National Monument  40.714527 -74.004474               7           9   

   num_entrance_fees  description_le

In [5]:
df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

In [6]:
df = df.drop_duplicates()

In [7]:
df.isna().sum()

park_name             0
park_code             0
states                0
designation           0
latitude              0
longitude             0
num_activities        0
num_topics            0
num_entrance_fees     0
description_length    0
has_camping           0
has_fishing           0
has_wildlife          0
has_biking            0
has_boating           0
dtype: int64

In [9]:
df["has_camping"] = df["has_camping"].astype(bool)
df["has_fishing"] = df["has_fishing"].astype(bool)
df["has_wildlife"] = df["has_wildlife"].astype(bool)
df["has_biking"] = df["has_biking"].astype(bool)
df["has_boating"] = df["has_boating"].astype(bool)

In [10]:
print(df.info())
print(df.describe())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 474 entries, 0 to 473
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   park_name           474 non-null    object 
 1   park_code           474 non-null    object 
 2   states              474 non-null    object 
 3   designation         474 non-null    object 
 4   latitude            474 non-null    float64
 5   longitude           474 non-null    float64
 6   num_activities      474 non-null    int64  
 7   num_topics          474 non-null    int64  
 8   num_entrance_fees   474 non-null    int64  
 9   description_length  474 non-null    int64  
 10  has_camping         474 non-null    bool   
 11  has_fishing         474 non-null    bool   
 12  has_wildlife        474 non-null    bool   
 13  has_biking          474 non-null    bool   
 14  has_boating         474 non-null    bool   
dtypes: bool(5), float64(2), int64(4), object(4)
memory usage:

In [11]:
df.to_csv("national_parks_dataset.csv", index=False)